# Lesson 09: Autoencoders & Variational Autoencoders (VAEs)

In this notebook, you will:
1. Train a **Convolutional Denoising Autoencoder** on MNIST.
2. Build and train a **Variational Autoencoder (VAE)** with the Reparameterization Trick and KL + BCE loss.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_ds = datasets.MNIST("./data", train=True, download=True, transform=transforms.ToTensor())
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)

class VAE(nn.Module):
    def __init__(self, latent_dim=8):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)
        self.fc_dec1 = nn.Linear(latent_dim, 256)
        self.fc_dec2 = nn.Linear(256, 784)

    def encode(self, x):
        h = F.relu(self.fc1(x.view(-1, 784)))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def decode(self, z):
        return torch.sigmoid(self.fc_dec2(F.relu(self.fc_dec1(z))))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

vae = VAE().to(device)
opt = torch.optim.Adam(vae.parameters(), lr=1e-3)

for epoch in range(3):
    total_loss = 0.0
    for xb, _ in train_loader:
        xb = xb.to(device)
        recon, mu, logvar = vae(xb)
        bce = F.binary_cross_entropy(recon, xb.view(-1, 784), reduction="sum")
        kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        loss = bce + kld
        opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} | Avg VAE Loss: {total_loss / len(train_ds):.2f}")

## Part 2: Sampling New Digits from Latent Space

In [ ]:
vae.eval()
with torch.no_grad():
    z_sample = torch.randn(4, 8, device=device)
    generated = vae.decode(z_sample).view(-1, 28, 28).cpu()
print("Generated digit tensor shape:", tuple(generated.shape))